In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn import preprocessing
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
#access and read dataset
dataset = pd.read_csv('/content/drive/MyDrive/job_salary_prediction_dataset.csv')

#Features of data
print("Original Features:", dataset.columns)

#Size of data
print("Size of dataset (rows, columns):", dataset.shape)


In [5]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
dataset = dataset.drop(columns =['remote_work'])

# Let's view the top few rows of the dataset
dataset.head()

# Let's show the data types of all the variables
print(dataset.dtypes)


In [ ]:
dataset.isnull().sum()


In [ ]:
#One-Hot Encoding convert colunm type from object to int
categorical_cols = dataset.select_dtypes(include=['object']).columns
dataset_encoded = pd.get_dummies(dataset, columns=categorical_cols, drop_first=True, dtype=int)

print(dataset_encoded.head())



In [ ]:
#Separating X and y

# The target variable (what we want to predict)
y = dataset_encoded['salary']

# The feature variables (what we use for prediction)
X = dataset_encoded.drop('salary', axis=1)

# --- Verification (Optional but Recommended) ---
print("--- Target Variable (y) ---")
print(y.head())
print(f"\nShape of y: {y.shape}")


print("\n--- Feature Matrix (X) ---")
print(X.head())
print(f"\nShape of X: {X.shape}")



In [ ]:
# Split the data into training (60%) and testing (40%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.4, random_state = 42)

print("--- Data Shapes After Splitting ---")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("-----------------------------------")
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

In [ ]:
# 1. Create an instance of the model
linear_model = LinearRegression()

# 2. Train the model using the training data
# The .fit() method is where the model "learns" from the data
print("Training the Linear Regression model...")
linear_model.fit(X_train, y_train)
print("Training complete!")

# The model is now trained and stored in the 'linear_model' variable.

# --- Evaluate the Model (from Step 8) to get predictions ---
print("Making predictions on the test set...")
lr_pred = linear_model.predict(X_test)


# --- NEW: Visualize the Model's Performance ---
print("Generating visualization...")

# Create a scatter plot of Actual vs. Predicted values
plt.figure(figsize=(10, 8))
sns.scatterplot(x=y_test, y=lr_pred, alpha=0.6)

# Add a line for perfect predictions (y=x)
# Find the min and max values for a consistent axis range
p1 = max(max(lr_pred), max(y_test))
p2 = min(min(lr_pred), min(y_test))
plt.plot([p1, p2], [p1, p2], 'r--', linewidth=2, label='Perfect Prediction')

# Add titles and labels for clarity
plt.title('Linear Regression: Actual vs. Predicted Salary', fontsize=16)
plt.xlabel('Actual Salary ($)', fontsize=12)
plt.ylabel('Predicted Salary ($)', fontsize=12)
plt.legend()
plt.grid(True)
plt.show()

# --- Optional: Display the performance metrics again for context ---
lr_mae = mean_absolute_error(y_test, lr_pred)
lr_r2 = r2_score(y_test, lr_pred)
print(f"\nLinear Regression R-squared (R²): {lr_r2:.4f}")
print(f"Linear Regression Mean Absolute Error (MAE): ${lr_mae:,.2f}")

In [ ]:
# --- Displaying the results of the training ---

# 1. Get the intercept
intercept = linear_model.intercept_
print(f"Model Intercept: ${intercept:,.2f}\n")

# 2. Get the coefficients
# We'll match them with their feature names to make them understandable
feature_names = X.columns  # Assuming X is your feature DataFrame from Step 3
coefficients = linear_model.coef_

# Create a DataFrame to view them clearly
coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefficients})

# Sort by the absolute value of the coefficient to see the most impactful features
coef_df['Absolute_Coefficient'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values(by='Absolute_Coefficient', ascending=False)
del coef_df['Absolute_Coefficient']

print("--- Model Coefficients (Top 10 Most Impactful) ---")
print(coef_df.head(10))



In [ ]:
# 1. Create an instance of the model
# We can set n_estimators, which is the number of "trees" in the forest. 100 is a good starting point.
# We also set random_state for reproducibility of the forest itself!
random_forest_model = RandomForestRegressor(n_estimators=100, random_state=42)

# 2. Train the model using the same training data
print("Training the Random Forest model... (This may take a few moments)")
random_forest_model.fit(X_train, y_train)
print("Training complete!")

# The trained model, with all its 100 trees, is now stored in the 'random_forest_model' variable.





# --- Evaluate the Model to get predictions ---
print("Making predictions on the test set...")
rf_pred = random_forest_model.predict(X_test)

# --- Create a figure with two subplots (side-by-side) ---
fig, axes = plt.subplots(1, 2, figsize=(20, 8))


# --- Plot 1: Actual vs. Predicted ---
sns.scatterplot(x=y_test, y=rf_pred, alpha=0.6, ax=axes[0])
p1 = max(max(rf_pred), max(y_test))
p2 = min(min(rf_pred), min(y_test))
axes[0].plot([p1, p2], [p1, p2], 'r--', linewidth=2)
axes[0].set_title('Random Forest: Actual vs. Predicted Salary', fontsize=16)
axes[0].set_xlabel('Actual Salary ($)', fontsize=12)
axes[0].set_ylabel('Predicted Salary ($)', fontsize=12)
axes[0].grid(True)


# --- Plot 2: Residual Plot ---
residuals = y_test - rf_pred
sns.scatterplot(x=rf_pred, y=residuals, alpha=0.6, ax=axes[1])
axes[1].axhline(y=0, color='r', linestyle='--', linewidth=2) # Add the y=0 line
axes[1].set_title('Random Forest: Residual Plot', fontsize=16)
axes[1].set_xlabel('Predicted Salary ($)', fontsize=12)
axes[1].set_ylabel('Residuals (Actual - Predicted) ($)', fontsize=12)
axes[1].grid(True)

plt.suptitle('Random Forest Performance Visualization', fontsize=20)
plt.show()

# --- Display performance metrics ---
rf_r2 = r2_score(y_test, rf_pred)
rf_mae = mean_absolute_error(y_test, rf_pred)
print(f"\nRandom Forest R-squared (R²): {rf_r2:.4f}")
print(f"Random Forest Mean Absolute Error (MAE): ${rf_mae:,.2f}")


In [ ]:
# 1. Create an instance of the model
# n_estimators is the number of trees in the sequence.
# learning_rate controls how much each tree contributes to the final prediction.
# random_state ensures reproducibility.
gradient_boosting_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)

# 2. Train the model using the same training data
print("Training the Gradient Boosting model... (This may also take a few moments)")
gradient_boosting_model.fit(X_train, y_train)
print("Training complete!")

# The trained sequential model is now stored in the 'gradient_boosting_model' variable.






# --- Evaluate the Model to get predictions ---
print("Making predictions on the test set...")
gb_pred = gradient_boosting_model.predict(X_test)

# --- Create a figure with two subplots (side-by-side) ---
fig, axes = plt.subplots(1, 2, figsize=(20, 8))


# --- Plot 1: Actual vs. Predicted ---
sns.scatterplot(x=y_test, y=gb_pred, alpha=0.6, ax=axes[0])
p1 = max(max(gb_pred), max(y_test))
p2 = min(min(gb_pred), min(y_test))
axes[0].plot([p1, p2], [p1, p2], 'r--', linewidth=2)
axes[0].set_title('Gradient Boosting: Actual vs. Predicted', fontsize=16)
axes[0].set_xlabel('Actual Salary ($)', fontsize=12)
axes[0].set_ylabel('Predicted Salary ($)', fontsize=12)
axes[0].grid(True)


# --- Plot 2: Residual Plot ---
residuals = y_test - gb_pred
sns.scatterplot(x=gb_pred, y=residuals, alpha=0.6, ax=axes[1])
axes[1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1].set_title('Gradient Boosting: Residual Plot', fontsize=16)
axes[1].set_xlabel('Predicted Salary ($)', fontsize=12)
axes[1].set_ylabel('Residuals (Actual - Predicted) ($)', fontsize=12)
axes[1].grid(True)

plt.suptitle('Gradient Boosting Performance Visualization', fontsize=20)
plt.show()

# --- Display performance metrics ---
gb_r2 = r2_score(y_test, gb_pred)
gb_mae = mean_absolute_error(y_test, gb_pred)
print(f"\nGradient Boosting R-squared (R²): {gb_r2:.4f}")
print(f"Gradient Boosting Mean Absolute Error (MAE): ${gb_mae:,.2f}")


In [ ]:
# --- 1. Predictions from Linear Regression ---
lr_pred = linear_model.predict(X_test)
lr_mae = mean_absolute_error(y_test, lr_pred)
lr_r2 = r2_score(y_test, lr_pred)

# --- 2. Predictions from Random Forest ---
rf_pred = random_forest_model.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)

# --- 3. Predictions from Gradient Boosting ---
gb_pred = gradient_boosting_model.predict(X_test)
gb_mae = mean_absolute_error(y_test, gb_pred)
gb_r2 = r2_score(y_test, gb_pred)


# --- Display the Results ---
print("--- Model Performance Evaluation ---")

print("\nLinear Regression:")
print(f"R-squared (R²): {lr_r2:.4f}")
print(f"Mean Absolute Error (MAE): ${lr_mae:,.2f}")

print("\nRandom Forest:")
print(f"R-squared (R²): {rf_r2:.4f}")
print(f"Mean Absolute Error (MAE): ${rf_mae:,.2f}")

print("\nGradient Boosting:")
print(f"R-squared (R²): {gb_r2:.4f}")
print(f"Mean Absolute Error (MAE): ${gb_mae:,.2f}")



In [ ]:
# 1. Create a dictionary with the results
results_data = {
    'Model': ['Linear Regression', 'Random Forest', 'Gradient Boosting'],
    'R-squared (R²)': [lr_r2, rf_r2, gb_r2],
    'Mean Absolute Error (MAE)': [lr_mae, rf_mae, gb_mae]
}

# 2. Convert the dictionary into a pandas DataFrame
results_df = pd.DataFrame(results_data)

# 3. Sort the DataFrame to rank the models
# We sort by R² descending (best first) and then by MAE ascending (best first)
results_df = results_df.sort_values(by=['R-squared (R²)', 'Mean Absolute Error (MAE)'], ascending=[False, True])

# 4. Reset the index so the ranks start from 0
results_df = results_df.reset_index(drop=True)


# --- Display the final scorecard ---
print("--- Model Comparison Scorecard ---")
print(results_df)



In [ ]:
# Let's assume 'gradient_boosting_model' was our best model
# You can easily swap this with 'random_forest_model' if it performed better
best_model = gradient_boosting_model

# 1. Get the feature importances from the trained model
importances = best_model.feature_importances_
feature_names = X.columns # Get the feature names from our X DataFrame

# 2. Create a DataFrame for better visualization
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})

# 3. Sort the DataFrame by importance in descending order
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# --- Display the Top 10 Most Important Features (The Table) ---
print("--- Top 10 Most Important Features for Predicting Salary ---")
print(feature_importance_df.head(10))


# --- Create the Visualization ---
plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df.head(10), palette='viridis')

plt.title('Top 10 Most Important Features for Salary Prediction', fontsize=16)
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

